In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import DataLoader
import yfinance as yf

**Contexto: Aplicação dos indicadores SMA, EMA e RSI, que se tornam indicadores de momentum e tendência para "prever" o fechamento do ativo**

**Média Móvel Simples (SMA – Simple Moving Average)média aritmética dos preços de fechamento de um ativo durante um determinado período.**

**Média Móvel Exponencial (EMA – Exponential Moving Average) A EMA dá mais peso aos dados recentes, tornando-a mais sensível a mudanças recentes nos preços**

**Índice de Força Relativa (RSI – Relative Strength Index) oscilador que captura e mede a velocidade e a mudança dos movimentos de preço**

In [ ]:
tickers = ["INTC", "GOOGL", "AMZN", "AMD"]
seq_len = 90
n_future = 7

def add_indicators(df, col, window=14):
    df[f'SMA_{window}'] = df[col].rolling(window).mean()
    df[f'EMA_{window}'] = df[col].ewm(span=window, adjust=False).mean()

    delta = df[col].diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(window).mean()
    avg_loss = loss.rolling(window).mean()

    rs = avg_gain / (avg_loss + 1e-8)
    df[f'RSI_{window}'] = 100 - (100/(1+rs))

    return df

dfs = []
for t in tickers:
    df = yf.download(t, start="2015-01-01", end="2025-01-01", progress=False)
    df = df[['Close']].rename(columns={'Close': f'{t}_Close'})
    df = add_indicators(df, col=f'{t}_Close')
    dfs.append(df)

data = pd.concat(dfs, axis=1).dropna()

X_raw = data.values
y_raw = data[[f"{t}_Close" for t in tickers]].values

**Normalização para preparar os dados para a LSTM**

In [ ]:
split_idx = int(0.8 * len(X_raw))
train_X_raw, test_X_raw = X_raw[:split_idx], X_raw[split_idx:]
train_y_raw, test_y_raw = y_raw[:split_idx], y_raw[split_idx:]

scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_train_scaled = scaler_X.fit_transform(train_X_raw)
X_test_scaled = scaler_X.transform(test_X_raw)

y_train_scaled = scaler_y.fit_transform(train_y_raw)
y_test_scaled = scaler_y.transform(test_y_raw)

In [ ]:
def create_sequences(X, y, seq_len):
    Xs, ys = [], []
    for i in range(seq_len, len(X)):
        Xs.append(X[i-seq_len:i])
        ys.append(y[i])
    return np.array(Xs), np.array(ys)

X_train_seq, y_train_seq = create_sequences(X_train_scaled, y_train_scaled, seq_len)
X_test_seq, y_test_seq = create_sequences(X_test_scaled, y_test_scaled, seq_len)


**Camada LSTM + GRU para capturar padrões temporais que tenha múltiplas variáveis**

In [ ]:
class LSTM_GRU_Model(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.lstm = nn.LSTM(input_size, 128, batch_first=True)
        self.gru = nn.GRU(128, 128, batch_first=True)
        self.fc = nn.Linear(128, output_size)

    def forward(self, x):
        out, _ = self.lstm(x)
        out, _ = self.gru(out)
        return self.fc(out[:, -1, :])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LSTM_GRU_Model(X_train_seq.shape[2], len(tickers)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

In [ ]:
train_loader = DataLoader(
    list(zip(torch.tensor(X_train_seq, dtype=torch.float32),
             torch.tensor(y_train_seq, dtype=torch.float32))),
    batch_size=128, shuffle=True
)

val_loader = DataLoader(
    list(zip(torch.tensor(X_test_seq, dtype=torch.float32),
             torch.tensor(y_test_seq, dtype=torch.float32))),
    batch_size=128, shuffle=False
)

In [ ]:
epochs = 90
train_losses, val_losses = [], []

for epoch in range(1, epochs + 1):
    model.train()
    train_l = []

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()

        train_l.append(loss.item())

    model.eval()
    val_l = []

    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            val_l.append(criterion(model(xb), yb).item())

    train_losses.append(np.mean(train_l))
    val_losses.append(np.mean(val_l))

    if epoch % 10 == 0:
        print(f"Época {epoch}: Train={train_losses[-1]:.6f} | Val={val_losses[-1]:.6f}")

model.eval()
with torch.no_grad():
    preds = model(torch.tensor(X_test_seq, dtype=torch.float32).to(device)).cpu().numpy()

pred_prices = scaler_y.inverse_transform(preds)
real_prices = scaler_y.inverse_transform(y_test_seq)

**avaliação do processo de aprendizado**

**Train Loss → erro nos dados de treino**

**Validation Loss → erro nos dados de teste**

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(train_losses, label='Train')
plt.plot(val_losses, label='Validation')
plt.legend(); plt.grid(); plt.show()

**Desempenho do modelo no conjunto deste teste com:**

**Modelo MAE = média dos erros absolutos entre os valores reais e os valores previstos**

**Modelo RMSE = raiz da média dos erros quadráticos entre valores reais e previstos = penaliza mais as previsões erradas**


In [ ]:
print(f"{'Ativo':<10} | {'MAE':<10} | {'RMSE':<10}")
print("-"*35)

fig, axes = plt.subplots(2,2, figsize=(15,10))
axes = axes.flatten()

for i, t in enumerate(tickers):
    mae = mean_absolute_error(real_prices[:, i], pred_prices[:, i])
    rmse = np.sqrt(mean_squared_error(real_prices[:, i], pred_prices[:, i]))

    print(f"{t:<10} | {mae:<10.2f} | {rmse:<10.2f}")

    axes[i].plot(real_prices[:, i], label='Real')
    axes[i].plot(pred_prices[:, i], '--', label='Previsto')
    axes[i].set_title(t)
    axes[i].legend()

plt.tight_layout()
plt.show()

**tendência esperada por ação no curto prazo (7 dias). estimativa simplificada**

In [ ]:
current_sequence = X_test_scaled[-seq_len:]
future_preds = []

for _ in range(n_future):
    inp = torch.tensor(current_sequence, dtype=torch.float32).unsqueeze(0).to(device)

    with torch.no_grad():
        pred = model(inp).cpu().numpy()

    future_preds.append(pred[0])
    next_step = current_sequence[-1].copy()
    next_step[:len(tickers)] = pred[0]
    current_sequence = np.vstack([current_sequence[1:], next_step])

future_prices = scaler_y.inverse_transform(np.array(future_preds))
df_future = pd.DataFrame(
    future_prices,
    columns=tickers,
    index=[f"Dia {i+1}" for i in range(n_future)]
)

print("\nProjeção futura:")
print(df_future.round(2))